# Crear les dades inicials amb els paràmetres inicials

In [76]:
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import numpy as np

In [77]:
def simulate_tour(tour_df:pd.DataFrame, players_df:pd.DataFrame, k, ksi, s, initial_elo, min_games, year_to_simulate): 
    assert tour_df.isna().sum().sum() == 0, f'nan values in tour_df\n{tour_df.isna().sum()}'

    tour_df = tour_df.sort_values(by='tourney_date', ascending=True)

    if year_to_simulate != 'Tots': 
        tour_df = tour_df[tour_df['tour_year']==year_to_simulate]

    all_players_dic = {player_id: {
            'player_id': player_id, 
            'elo_rating': initial_elo, 
            'elo_clay_rating': initial_elo, 
            'elo_hard_rating': initial_elo, 
            'elo_grass_rating': initial_elo, 
            'elo_carpet_rating': initial_elo, 
            'elo_unknown_rating': initial_elo,
            'n_games': 0,
            'last_game': None,
            'n_wins': 0,
            'n_losses': 0
        } for player_id in players_df['player_id']}

    elo_history_list = []
    ranking_historic = pd.DataFrame()
    year = ''
    date = tour_df['tourney_date'].iloc[0]
    for _, m in tour_df.iterrows():
        
        # La primera vegada no hi entra
        if date != m['tourney_date']: 
            # Guardem el ranking block en un dataframe
            ranking_block_df = pd.DataFrame.from_dict(ranking_block, orient='index')\
                .sort_values(by='elo_rating')\
                
            # Afegim la columna 'date', i rank i concatenem al ranking_historic
            ranking_block_df['date'] = date
            ranking_block_df['rank'] = ranking_block_df.index + 1
            ranking_historic = pd.concat([ranking_historic, ranking_block_df], ignore_index=True)

            # Actualitzem la data
            date = m['tourney_date']

        # Si l'any canvia, canviem el ranking block amb els jugadors del nou tour. 
        if year != m['match_year']: # La primera vegada si que hi entra.
            year = m['match_year']
            print(year)
            print(f'Year: {year}')
            # Combine winner_id and loser_id, then extract unique IDs
            tour_p_id = pd.concat([
                tour_df[tour_df['match_year'] == year]['winner_id'],
                tour_df[tour_df['match_year'] == year]['loser_id']
            ]).unique()

            ranking_block = [{
                p_id: all_players_dic[p_id]['elo_rating'] for p_id in tour_p_id
            }]


        # Update elo ratings
        match s:
            case 'delta':
                Sw = 1
                Sl = 0

            case 'thirds': 
                if m['best_of'] != m['num_sets'] or m['best_of'] == 1:
                    Sw = 1
                    Sl = 0
                else: 
                    Sw = 2/3
                    Sl = 1/3

        
        # Algorisme per calcular elo-ratings
        winner_id = m['winner_id']
        loser_id = m['loser_id']
        elo_surface = f'elo_{m['surface'].lower()}_rating'
        match_date = m['tourney_date']
        surface = m['surface']

        old_wr =  all_players_dic[winner_id]['elo_rating']
        old_lr = all_players_dic[loser_id]['elo_rating']
        # Surface
        old_slr = all_players_dic[winner_id][elo_surface]
        old_swr = all_players_dic[loser_id][elo_surface]

        
        mu_w = 1 / (1 + pow(10, -(old_wr - old_lr)/ksi))
        mu_l = 1 / (1 + pow(10, -(old_lr - old_wr)/ksi))
        # Surface
        mu_sw = 1 / (1 + pow(10, -(old_swr - old_slr)/ksi))
        mu_sl = 1 / (1 + pow(10, -(old_slr - old_swr)/ksi))

        # Actualitzar els valors dels elo-ratings dels jugadors. 
        winner_new_elo = old_wr + k*(Sw - mu_w)
        loser_new_elo = old_lr + k*(Sl - mu_l)
        all_players_dic[winner_id]['elo_rating'] = winner_new_elo
        all_players_dic[loser_id]['elo_rating'] = loser_new_elo
        # Surface
        all_players_dic[winner_id][elo_surface] = old_swr + k*(Sw - mu_sw)
        all_players_dic[loser_id][elo_surface] = old_slr + k*(Sl - mu_sl)

        all_players_dic[loser_id]['n_games'] += 1
        all_players_dic[loser_id]['last_game'] = match_date
        all_players_dic[winner_id]['n_games'] += 1
        all_players_dic[winner_id]['last_game'] = match_date
        all_players_dic[winner_id]['n_wins'] += 1
        all_players_dic[loser_id]['n_losses'] += 1
        assert all_players_dic[winner_id]['n_games'] == all_players_dic[winner_id]['n_wins'] + all_players_dic[winner_id]['n_losses'],\
            f"Error: n_games != n_wins + n_losses -> {all_players_dic[winner_id]['n_games']} != {all_players_dic[winner_id]['n_wins']} + {all_players_dic[winner_id]['n_losses']}"

        # Històric
        winner_history = {
            'player_id': winner_id,
            'date': match_date,
            'elo_rating': winner_new_elo
        }
        loser_history = {
            'player_id': loser_id,
            'date': match_date,
            'elo_rating': loser_new_elo
        }

        elo_history_list.append(winner_history)
        elo_history_list.append(loser_history)

    # TODO: 
    # Passar all_players_dic a dataframe
    # Fer merge amb el df de tots els jugadors per tenir totes les stats
    # Filtrar per minim de partits
    # Crear columna rank
    # Passar la llista de diccionaris amb l'historic a un dataframe i canviar format de la data
    
    

    players_simulated: pd.DataFrame = pd.DataFrame.from_dict(all_players_dic, orient='index')

    ranking: pd.DataFrame = players_df.merge(players_simulated, how='left', on='player_id')\
        .sort_values(by='elo_rating', ascending=False)\
        .reset_index(drop=True)\
        .drop(columns=['elo_unknown_rating'])\
        .round(0)\
        .rename(columns={
            'elo_rating': 'Elo Rating', 
            'elo_clay_rating': 'Clay Elo Rating', 
            'elo_hard_rating': 'Hard Elo Rating', 
            'elo_grass_rating': 'Grass Elo Rating', 
            'elo_carpet_rating': 'Carpet Elo Rating'
        })


    ranking_filtered = ranking[(ranking['n_games']>min_games)]# & (ranking['last_game'] > '2023-01-01')].reset_index(drop=True)

    ranking_filtered['rank'] = ranking_filtered.index + 1

    elo_history_df = pd.DataFrame(elo_history_list)\
                        .astype({'date': 'datetime64[ns]'})

    return ranking_filtered, elo_history_df, ranking_historic


In [78]:
def simulate_tour_optimized(tour_df:pd.DataFrame, players_df:pd.DataFrame, k, xi, s, initial_elo, min_games, year_to_simulate): 
    
    assert tour_df.isna().sum().sum() == 0, f'nan values in tour_df\n{tour_df.isna().sum()}'

    tour_df = tour_df.sort_values(by='tourney_date', ascending=True)

    if year_to_simulate != 'Tots': 
        tour_df = tour_df[tour_df['tour_year']==year_to_simulate]

    all_players_dic = {player_id: {
            'player_id': player_id, 
            'elo_rating': initial_elo, 
            'elo_clay_rating': initial_elo, 
            'elo_hard_rating': initial_elo, 
            'elo_grass_rating': initial_elo, 
            'elo_carpet_rating': initial_elo, 
            'elo_unknown_rating': initial_elo,
            'n_games': 0,
            'last_game': None,
            'n_wins': 0,
            'n_losses': 0
        } for player_id in players_df['player_id']}
    
    elo_rankings_df = pd.DataFrame()
    elo_history_list = []
    unique_years = np.sort(tour_df['tour_year'].unique())
    
    # Bucle principal
    for year in unique_years:
        print(year)
        
        # Jugadors unics del year_block
        unique_p_ids = pd.concat([
                tour_df[tour_df['tour_year'] == year]['winner_id'],
                tour_df[tour_df['tour_year'] == year]['loser_id']
            ]).unique()

        # Dates uniques del year_block
        unique_dates = np.sort(tour_df[tour_df['tour_year']==year]['tourney_date'].unique())
        print(unique_dates)
        for date in unique_dates:             
            # Date block per guardar resultats i concatenar al final
            date_block_dict = {player_id: {
                'player_id': player_id,
                'elo_rating': all_players_dic[player_id]['elo_rating'],
                'elo_clay_rating': all_players_dic[player_id]['elo_clay_rating'],
                'elo_hard_rating': all_players_dic[player_id]['elo_hard_rating'],
                'elo_grass_rating': all_players_dic[player_id]['elo_grass_rating'],
                'elo_carpet_rating': all_players_dic[player_id]['elo_carpet_rating'],
                'elo_unknown_rating': all_players_dic[player_id]['elo_unknown_rating'],
            } for player_id in unique_p_ids}
                
            # Date block sobre el qual iterarem
            date_block_df: pd.DataFrame = tour_df[(tour_df['tourney_date'] == date) & (tour_df['tour_year'] == year)] # Fer el match del year no cal pk la date ja es més restrictiu
            if date_block_df.empty:
                print(f"date_block_df is empty for date {date} and year {year}")
                exit(0)
            for _, m in date_block_df.iterrows(): 

                # Obtenir algunes dades
                elo_surface = f'elo_{m['surface'].lower()}_rating'
                old_wr = date_block_dict[m['winner_id']]['elo_rating']
                old_swr = date_block_dict[m['winner_id']][elo_surface]
                old_lr = date_block_dict[m['loser_id']]['elo_rating']
                old_slr = date_block_dict[m['loser_id']][elo_surface]

                winner_id = m['winner_id']
                loser_id = m['loser_id']
                elo_surface = f'elo_{m['surface'].lower()}_rating'
                match_date = m['tourney_date']
                surface = m['surface']
                
                match s:
                    case 'delta':
                        Sw = 1
                        Sl = 0

                    case 'thirds': 
                        if m['best_of'] != m['num_sets'] or m['best_of'] == 1:
                            Sw = 1
                            Sl = 0
                        else: 
                            Sw = 2/3
                            Sl = 1/3

                # Algorisme #
                mu_w = 1 / (1 + pow(10, -(old_wr - old_lr)/xi))
                mu_l = 1 / (1 + pow(10, -(old_lr - old_wr)/xi))
                # Surface
                mu_sw = 1 / (1 + pow(10, -(old_swr - old_slr)/xi))
                mu_sl = 1 / (1 + pow(10, -(old_slr - old_swr)/xi))

                # Actualitzar els valors dels elo-ratings dels jugadors. 
                winner_new_elo = old_wr + k*(Sw - mu_w)
                loser_new_elo = old_lr + k*(Sl - mu_l)
                winner_new_s_elo = old_swr + k*(Sw - mu_sw)
                loser_new_s_elo = old_slr + k*(Sl - mu_sl)

                # Guardem resultats al date_block
                date_block_dict[winner_id]['elo_rating'] = winner_new_elo
                date_block_dict[loser_id]['elo_rating'] = loser_new_elo
                date_block_dict[winner_id][elo_surface] = winner_new_s_elo
                date_block_dict[loser_id][elo_surface] = loser_new_s_elo

                # Guardem resultats al diccionari de jugadors
                all_players_dic[winner_id]['elo_rating'] = winner_new_elo
                all_players_dic[loser_id]['elo_rating'] = loser_new_elo
                all_players_dic[winner_id][elo_surface] = winner_new_s_elo
                all_players_dic[loser_id][elo_surface] = loser_new_s_elo

                # Stats #
                all_players_dic[loser_id]['n_games'] += 1
                all_players_dic[loser_id]['last_game'] = match_date
                all_players_dic[winner_id]['n_games'] += 1
                all_players_dic[winner_id]['last_game'] = match_date
                all_players_dic[winner_id]['n_wins'] += 1
                all_players_dic[loser_id]['n_losses'] += 1
                assert all_players_dic[winner_id]['n_games'] == all_players_dic[winner_id]['n_wins'] + all_players_dic[winner_id]['n_losses'],\
                    f"Error: n_games != n_wins + n_losses -> {all_players_dic[winner_id]['n_games']} != {all_players_dic[winner_id]['n_wins']} + {all_players_dic[winner_id]['n_losses']}"


                # Històric (com un jugador pot jugar més d'un partit en un mateix dia, al data_block només s'hi guardarà l'elo amb què acaba el dia)
                # Per al plot creo una llista amb tots els elo-ratings de cada jugador
                winner_history = {
                    'player_id': winner_id,
                    'date': match_date,
                    'elo_rating': winner_new_elo
                }
                loser_history = {
                    'player_id': loser_id,
                    'date': match_date,
                    'elo_rating': loser_new_elo
                }

                elo_history_list.append(winner_history)
                elo_history_list.append(loser_history)

            # Concatenem el ranking block en un dataframe
            ranking_block_df = pd.DataFrame.from_dict(date_block_dict, orient='index')\
                .sort_values(by='elo_rating')
            
            # Afegim columnes date i rank i concatenem al ranking_historic
            ranking_block_df['date'] = date
            ranking_block_df['rank'] = ranking_block_df.index + 1
            elo_rankings_df = pd.concat([elo_rankings_df, ranking_block_df], ignore_index=True)


    # Passar all_players_dic a dataframe
    all_players_df: pd.DataFrame = pd.DataFrame.from_dict(all_players_dic, orient='index')

    # Fer merge amb el df de tots els jugadors per tenir totes les stats
    ranking: pd.DataFrame = players_df.merge(all_players_df, how='left', on='player_id')\
        .sort_values(by='elo_rating', ascending=False)\
        .reset_index(drop=True)\
        .drop(columns=['elo_unknown_rating'])\
        .round(0)\
        .rename(columns={
            'elo_rating': 'Elo Rating', 
            'elo_clay_rating': 'Clay Elo Rating', 
            'elo_hard_rating': 'Hard Elo Rating', 
            'elo_grass_rating': 'Grass Elo Rating', 
            'elo_carpet_rating': 'Carpet Elo Rating'
        })

    # Filtrar per minim de partits
    ranking_filtered = ranking[(ranking['n_games']>min_games)]# & (ranking['last_game'] > '2023-01-01')].reset_index(drop=True)

    # Crear columna rank
    ranking_filtered['rank'] = ranking_filtered.index + 1

    # Passar la llista de diccionaris amb l'historic a un dataframe i canviar format de la data
    elo_history_df = pd.DataFrame(elo_history_list)\
                        .astype({'date': 'datetime64[ns]'})
    
    return ranking_filtered, elo_history_df, elo_rankings_df
    



In [79]:
k = 24
ksi = 400
s = 'delta'
initial_elo = 1500
min_games = 30
year_to_simulate = 'Tots'

In [80]:
atp_matches_df = pd.read_csv('../web_data/clean_atp_matches.csv')
atp_players_df = pd.read_csv('../web_data/clean_atp_players.csv')
wta_matches_df = pd.read_csv('../web_data/clean_wta_matches.csv')
wta_players_df = pd.read_csv('../web_data/clean_wta_players.csv')
assert atp_matches_df.isna().sum().sum() == 0, f'nan values in matches\n{atp_matches_df.isna().sum()}'
assert atp_players_df.isna().sum().sum() == 0, f'nan values in players\n{atp_players_df.isna().sum()}'
assert atp_matches_df.isna().sum().sum() == 0, f'nan values in matches\n{wta_matches_df.isna().sum()}'
assert atp_players_df.isna().sum().sum() == 0, f'nan values in players\n{wta_players_df.isna().sum()}'

In [81]:
atp_ranking, atp_elo_history, atp_ranking_historic = simulate_tour_optimized(atp_matches_df, atp_players_df, k, ksi, s, initial_elo, min_games, year_to_simulate)
atp_ranking.index += 1
wta_ranking, wta_elo_history, wta_ranking_historic = simulate_tour_optimized(wta_matches_df, wta_players_df, k, ksi, s, initial_elo, min_games, year_to_simulate)
wta_ranking.index += 1

1970
['1970-01-04' '1970-01-10' '1970-01-19' '1970-01-26' '1970-01-28'
 '1970-02-02' '1970-02-09' '1970-02-12' '1970-02-13' '1970-02-15'
 '1970-02-20' '1970-02-25' '1970-03-02' '1970-03-04' '1970-03-09'
 '1970-03-14' '1970-03-16' '1970-03-20' '1970-03-22' '1970-03-23'
 '1970-03-24' '1970-03-25' '1970-03-28' '1970-03-29' '1970-04-03'
 '1970-04-04' '1970-04-05' '1970-04-06' '1970-04-10' '1970-04-11'
 '1970-04-13' '1970-04-15' '1970-04-17' '1970-04-21' '1970-04-26'
 '1970-05-01' '1970-05-03' '1970-05-06' '1970-05-07' '1970-05-08'
 '1970-05-11' '1970-05-12' '1970-05-13' '1970-05-16' '1970-05-22'
 '1970-06-01' '1970-06-03' '1970-06-06' '1970-06-07' '1970-06-08'
 '1970-06-11' '1970-06-12' '1970-06-13' '1970-06-14' '1970-06-15'
 '1970-06-22' '1970-07-05' '1970-07-06' '1970-07-11' '1970-07-13'
 '1970-07-14' '1970-07-16' '1970-07-18' '1970-07-19' '1970-07-20'
 '1970-07-27' '1970-08-01' '1970-08-02' '1970-08-03' '1970-08-09'
 '1970-08-12' '1970-08-14' '1970-08-17' '1970-08-19' '1970-08-26'
 '197

/var/folders/s3/s4gmfm1j23xg8_5fnbsqlflm0000gp/T/ipykernel_54722/3891256993.py:169: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ranking_filtered['rank'] = ranking_filtered.index + 1


1968
['1967-12-25' '1967-12-28' '1968-01-01' '1968-01-02' '1968-01-08'
 '1968-01-16' '1968-01-19' '1968-01-22' '1968-01-26' '1968-01-28'
 '1968-01-29' '1968-02-05' '1968-02-09' '1968-02-12' '1968-02-16'
 '1968-02-19' '1968-02-20' '1968-02-26' '1968-02-29' '1968-03-01'
 '1968-03-04' '1968-03-11' '1968-03-17' '1968-03-18' '1968-03-21'
 '1968-03-25' '1968-03-26' '1968-04-01' '1968-04-02' '1968-04-08'
 '1968-04-14' '1968-04-15' '1968-04-16' '1968-04-17' '1968-04-22'
 '1968-04-25' '1968-04-29' '1968-05-05' '1968-05-06' '1968-05-18'
 '1968-05-20' '1968-05-21' '1968-05-22' '1968-05-23' '1968-05-24'
 '1968-05-25' '1968-05-27' '1968-05-30' '1968-06-03' '1968-06-04'
 '1968-06-06' '1968-06-10' '1968-06-14' '1968-06-16' '1968-06-17'
 '1968-06-24' '1968-07-01' '1968-07-04' '1968-07-08' '1968-07-14'
 '1968-07-15' '1968-07-22' '1968-07-23' '1968-07-25' '1968-07-29'
 '1968-07-31' '1968-08-05' '1968-08-06' '1968-08-08' '1968-08-10'
 '1968-08-11' '1968-08-12' '1968-08-13' '1968-08-15' '1968-08-17'
 '196

/var/folders/s3/s4gmfm1j23xg8_5fnbsqlflm0000gp/T/ipykernel_54722/3891256993.py:169: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ranking_filtered['rank'] = ranking_filtered.index + 1


In [82]:
atp_elo_history.to_csv('../web_data/atp_initial_elo_history.csv')
atp_ranking.to_csv('../web_data/atp_initial_ranking.csv')
wta_elo_history.to_csv('../web_data/wta_initial_elo_history.csv')
wta_ranking.to_csv('../web_data/wta_initial_ranking.csv')

In [83]:
atp_ranking_historic

,player_id,elo_rating,elo_clay_rating,elo_hard_rating,elo_grass_rating,elo_carpet_rating,elo_unknown_rating,date,rank
0,100058,1455.229261,1500.000000,1500.000000,1455.229261,1500.000000,1500.0,1970-01-04,100059
1,100084,1476.800015,1500.000000,1500.000000,1476.800015,1500.000000,1500.0,1970-01-04,100085
2,211204,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.0,1970-01-04,211205
3,211202,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.0,1970-01-04,211203
4,104418,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.0,1970-01-04,104419
...,...,...,...,...,...,...,...,...,...
1677282,126203,1915.372614,1717.192042,1866.265463,1636.172181,1500.000000,1500.0,2024-12-18,126204
1677283,100644,1959.480712,1889.616216,1909.891278,1661.708645,1500.000000,1500.0,2024-12-18,100645
1677284,207989,2015.639491,1916.387661,1928.491960,1728.682333,1500.000000,1500.0,2024-12-18,207990
1677285,104925,2083.319194,1999.464391,2083.287347,1965.354036,1565.236855,1500.0,2024-12-18,104926
